# Figure 5.6: Contours for the z-transform

*The Discrete Algebra of the Fourier Transform*, Gabriel Peyré.

The notch-filter transfer function evaluated on a unit circle and an outward logarithmic spiral. Both contours lie in the causal region of convergence; leaving the unit circle changes the depth of the notches.

[Read this figure in the book](https://mathematical-tours.github.io/daft-sources/DAFT-EN.pdf#page=120).

## Prepare the example

Run the cells in order in Jupyter or Google Colab. The first cell installs missing Python packages and downloads the figure sources and input data once. Subsequent runs use the cached download. Python 3.10 or later is required.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import urllib.request
import zipfile

packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "PIL": "Pillow", "sympy": "sympy"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

archive_url = os.environ.get("DAFT_FIGURE_ARCHIVE", 'https://mathematical-tours.github.io/daft-sources/downloads/daft-figures-2e843bbe89f0.zip')
archive_sha = '2e843bbe89f0e274849b6896eb16bd9f7686521a8ca98f633240bfc06ea2d5b1'
workspace = Path(os.environ.get("DAFT_FIGURE_WORKDIR", "daft-figure-files")) / archive_sha[:12]
cache = workspace / "sources.zip"
workspace.mkdir(parents=True, exist_ok=True)
data = cache.read_bytes() if cache.exists() else urllib.request.urlopen(archive_url, timeout=60).read()
if hashlib.sha256(data).hexdigest() != archive_sha:
    raise ValueError("The figure-source download is incomplete or has changed. Download it again.")
cache.write_bytes(data)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(workspace)
ROOT = (workspace / "daft-figures").resolve()
__file__ = str(ROOT / 'figures-code/chapter-related-transforms/z-transform-contours/generate.py')
print("Figure sources ready.")


## Generate the figure

This is the plotting code used by the book. Change its parameters and rerun this cell, then the display cell below, to explore the example.

In [ ]:
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[3]
sys.path.insert(0,str(ROOT/"tools"))
sys.path.insert(0,str(ROOT/"listings"))
from figure_style import *
from related_transform_data import notch,notch_impulse
from chirp import czt
n=128;k=np.arange(n+1);theta=2*np.pi*k/n
contours=[np.exp(1j*theta),np.exp((.1+2j*np.pi)*k/n)]
fig,axes=plt.subplots(1,2,figsize=(9,3.8))
for contour,label,color,style in zip(contours,['unit circle','outward spiral'],[TEAL,RED],['-','--']):
    axes[0].plot(contour.real,contour.imag,color=color,ls=style,label=label)
    axes[1].plot(k/n,abs(notch(contour)),color=color,ls=style,label=label)
    # Finite impulse-response evaluation agrees with the rational expression up to the tail.
    values=czt(notch_impulse(n),contour[1],n+1)
    np.testing.assert_allclose(values,notch(contour),atol=2e-6,rtol=2e-6)
zeros=np.exp(1j*np.array([-1,1])*np.pi/4);poles=.9*zeros
axes[0].plot(zeros.real,zeros.imag,'o',mfc='white',mec=BLUE,ms=6,label='zeros')
axes[0].plot(poles.real,poles.imag,'x',color=INK,ms=6,label='poles')
axes[0].set(aspect='equal',xlim=(-1.18,1.18),ylim=(-1.18,1.18),title='Evaluation contours')
axes[1].set(xlim=(0,1),ylim=(-.025,1.17),title='Transfer-function magnitude');axes[0].legend(fontsize=8,loc='lower left');axes[1].legend(fontsize=9,loc='lower center')
axes_box(axes[0],r'$\operatorname{Re}z$',r'$\operatorname{Im}z$');axes_box(axes[1],r'angle / $2\pi$',r'$|H(z)|$')
save(fig,'chapter-related-transforms','z-transform-contours')


## Display and save

In [ ]:
from IPython.display import Image, FileLink, display

display(Image(filename=str(ROOT / 'build/figure-previews/chapter-related-transforms/z-transform-contours.png')))
display(FileLink(os.path.relpath(ROOT / 'figures/chapter-related-transforms/z-transform-contours/figure.pdf', Path.cwd()),
                 result_html_prefix="Download the vector PDF: "))
